*Runs on Kaggle: attach the repo snapshot as an input Dataset and a GPU accelerator.*


# vn-tarot-llm — SFT fine-tuning

Fine-tunes Qwen3-1.7B on the frozen Vietnamese tarot SFT corpus (`datasets/filtered_core.jsonl`, 11.5k rows). Implements plan waves **W4.1–W4.3**. Works identically across Kaggle / vast.ai / Colab; only the setup cell differs.

In [ ]:

# KAGGLE SETUP — one-time preparation of the repo snapshot.
# 1. Upload this repository (without data/, .venv/, .cache/) as a PRIVATE Kaggle
#    Dataset named e.g. "<your-user>/tfvn-tarot-repo".
# 2. In the notebook's Input panel, attach that dataset.
# 3. Attach a GPU accelerator (P100 / T4 x2 / A100).
import os, shutil, sys

CANDIDATES = ["/kaggle/input/" + d for d in os.listdir("/kaggle/input")]
repo_src = next((c for c in CANDIDATES
                 if os.path.exists(os.path.join(c, "scripts/train_sft.py"))), None)
assert repo_src, ("repository snapshot not found among Kaggle inputs; "
                  f"looked in: {CANDIDATES}")

WORK = "/kaggle/working/tfvn-tarot"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(repo_src, WORK, ignore=shutil.ignore_patterns(
    ".git", ".venv", ".cache", "__pycache__", "data"))
os.chdir(WORK)
sys.path.insert(0, "src")

CORE = os.path.join(WORK, "datasets/filtered_core.jsonl")
BULK = os.path.join(WORK, "datasets/filtered_bulk.jsonl")
for p in (CORE, BULK):
    assert os.path.exists(p), f"missing dataset artifact: {p}"
n_rows = sum(1 for _ in open(CORE))
print(f"repo -> {WORK} | core rows: {n_rows}")


In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q torch==2.13.0 transformers==5.14.1 peft==0.20.0 trl==1.9.2 \
    accelerate==1.14.0 datasets==5.0.1 bitsandbytes==0.50.0 safetensors==0.8.0 einops scipy


In [ ]:

# Verify the stack BEFORE touching weights (plan W4.1 acceptance).
import torch, transformers, peft, trl
print("torch", torch.__version__, "| tf", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert torch.cuda.is_available(), "No CUDA GPU visible — select a GPU runtime."
name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} | CC {cc} | VRAM {vram:.1f} GB")
assert cc[0] >= 8, f"CC {cc}: bf16 requires CC >= 8 (Turing/Ampere+)."
bf16_ok = torch.cuda.is_bf16_supported()
major_gate = cc[0] >= 8
assert major_gate, "Plan gate: trust get_device_capability, not is_bf16_supported()."
print("bf16:", bf16_ok, "| capability-gate:", major_gate)
# v5 API surface used by train_sft.py must exist
from transformers import Trainer  # noqa
import inspect
assert "processing_class" in inspect.signature(Trainer.__init__).parameters, \
    "transformers v5 'processing_class=' missing — wrong version installed?"
print("STACK OK")


In [ ]:

# Baseline run (plan W4.2). Override here for ablations.
EPOCHS = 2          # stopping rule
DATA = CORE         # path to filtered_core.jsonl (set by setup cell)
MODEL = "Qwen/Qwen3-1.7B"
OUT = "artifacts/sft_r32"
print(f"training {MODEL} on {DATA} -> {OUT} for {EPOCHS} epochs")


In [ ]:

!python scripts/train_sft.py \
    --model "{MODEL}" \
    --data "{DATA}" \
    --out "{OUT}" \
    --epochs {EPOCHS}


In [ ]:

# Optional ablation bracket (plan W4.3): rank x data scale (~6 runs).
RUN_ABLATION = False
if RUN_ABLATION:
    ABL = [(r, CORE, f"artifacts/sft_r{r}_core") for r in (16, 32, 64)] + \
          [(r, BULK, f"artifacts/sft_r{r}_bulk") for r in (16, 32, 64)]
    print("\n".join(a[2] for a in ABL))


In [ ]:

if RUN_ABLATION:
    for r, data_path, out_dir in ABL:
        !python scripts/train_sft.py --model Qwen/Qwen3-1.7B \
            --data "{data_path}" --out "{out_dir}" \
            --epochs 2 --lora-r {r} --lora-alpha {r}


In [ ]:

# Post-training verification: adapter exists, metadata sane, loss recorded.
import json, glob, os
adapters = sorted(glob.glob("artifacts/*/best") + glob.glob("artifacts/*/smoke_final"))
assert adapters, "no trained adapter found under artifacts/"
latest = max(adapters, key=os.path.getmtime)
meta_path = os.path.join(os.path.dirname(latest), "run_meta.json")
meta = json.load(open(meta_path))
print("adapter :", latest)
print("loss    :", meta.get("final_loss"))
print("dtype   :", meta.get("dtype"), "| device:", meta.get("device"))
assert meta.get("final_loss") is not None, "training produced no loss record"
trip = os.path.join(os.path.dirname(latest), "orientation_tripwire_epoch1.json")
if os.path.exists(trip):
    t = json.load(open(trip))
    print(f"tripwire: rate={t['rate']:.2%} halt={t['halt']}")
    assert not t["halt"], "orientation tripwire fired — return to C2/C3"
print("RUN VERIFIED")


## Package outputs

In [ ]:

import shutil
# /kaggle/working persists as the run's output — keep it small (adapters only).
shutil.rmtree("/kaggle/working/artifacts/sft_r32/checkpoint-*", ignore_errors=True)
print("output size:")
os.system("du -sh /kaggle/working/artifacts/* 2>/dev/null")
